In [36]:
#  Setup and Imports
import pandas as pd
import pyodbc
from src.io.sql_materializer import load_temp_master_fact
from src.utils.db_connector import get_pyodbc_conn_string
print("Setup complete. Functions and connection utilities loaded.")

Setup complete. Functions and connection utilities loaded.


In [5]:
# Reading data from sql
def read_from_bronze(table_names: list) -> dict:
    print("\n--- Reading Data from Bronze Schema into Pandas Memory ---")
    conn_str = get_pyodbc_conn_string()
    cnxn = pyodbc.connect(conn_str)

    dataframes = {}
    for table_name in table_names:
        print(f"Reading table: {table_name}...")
        sql_query = f"SELECT * FROM bronze.{table_name}"
        dataframes[table_name] = pd.read_sql(sql_query, cnxn)
        print(f"Loaded {len(dataframes[table_name]):,} rows from bronze.{table_name}")

    cnxn.close()
    return dataframes

CORE_TABLES = [
    'orders_raw', 'customers_raw', 'order_items_raw', 'reviews_raw',
    'products_raw', 'category_trans_raw', 'payments_raw', 'sellers_raw',
    'geolocation_raw'
]
try:
    dfss = read_from_bronze(CORE_TABLES)
    print("\n✅ Successfully loaded all 9 tables into Pandas DataFrames (dfs dictionary).")
    print("DataFrames loaded: ", dfss.keys())
except Exception as e:
    print(f"❌ FAILED to read from SQL Bronze. Error: {e}")




--- Reading Data from Bronze Schema into Pandas Memory ---
Reading table: orders_raw...


C:\Users\Ayush\AppData\Local\Temp\ipykernel_1936\3508968805.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  dataframes[table_name] = pd.read_sql(sql_query, cnxn)


Loaded 99,441 rows from bronze.orders_raw
Reading table: customers_raw...
Loaded 99,441 rows from bronze.customers_raw
Reading table: order_items_raw...
Loaded 225,300 rows from bronze.order_items_raw
Reading table: reviews_raw...
Loaded 198,448 rows from bronze.reviews_raw
Reading table: products_raw...
Loaded 65,902 rows from bronze.products_raw
Reading table: category_trans_raw...
Loaded 142 rows from bronze.category_trans_raw
Reading table: payments_raw...
Loaded 207,772 rows from bronze.payments_raw
Reading table: sellers_raw...
Loaded 6,190 rows from bronze.sellers_raw
Reading table: geolocation_raw...
Loaded 2,000,326 rows from bronze.geolocation_raw

✅ Successfully loaded all 9 tables into Pandas DataFrames (dfs dictionary).
DataFrames loaded:  dict_keys(['orders_raw', 'customers_raw', 'order_items_raw', 'reviews_raw', 'products_raw', 'category_trans_raw', 'payments_raw', 'sellers_raw', 'geolocation_raw'])


In [6]:
# orders Raw cleaning
dfs=dfss
orders_raw_df=dfs['orders_raw']
orders_raw_df.head()
orders_raw_df.isnull().sum()

def check_timestamp_errors(df: pd.DataFrame) -> pd.Series:
    purchase_after_approved = (df['order_purchase_timestamp'] > df['order_approved_at'])
    approved_after_carrier = (df['order_approved_at'] > df['order_delivered_carrier_date'])

    carrier_after_customer = (df['order_delivered_carrier_date'] > df['order_delivered_customer_date'])
    error_mask = (purchase_after_approved.fillna(False)) | (approved_after_carrier.fillna(False)) | (carrier_after_customer.fillna(False))
    return error_mask

error_rows=check_timestamp_errors(orders_raw_df)
total_errors=error_rows.sum()
print("Total logical timestamp error found : ", total_errors)

def clean_timestamp_data(df: pd.DataFrame, error_mask: pd.Series)->pd.DataFrame:
    df_cleaned=df[~error_mask]
    deleted_count=error_mask.sum()

    print(f"Total logical errors dropped: {deleted_count:,}")
    print(f"Final Row Count: {len(df_cleaned):,}")
    return df_cleaned

orders_raw_df=clean_timestamp_data(orders_raw_df,error_rows)

Total logical timestamp error found :  1382
Total logical errors dropped: 1,382
Final Row Count: 98,059


In [7]:
# customers Raw cleaning
customer_raw_df=dfs['customers_raw']
customer_raw_df.head()
customer_raw_df.isnull().sum()

print("Total Unique customer : ",customer_raw_df['customer_unique_id'].nunique())
print("Total  customer Entry : ",len(customer_raw_df))

def customer_cleaning(df: pd.DataFrame)->pd.DataFrame:
    if df['customer_unique_id'].isnull().sum() > 0:
        df=df.dropna(subset=['customer_unique_id'])

    df['customer_city'] = df['customer_city'].str.strip().str.title()
    df['customer_state'] = df['customer_state'].str.strip().str.upper()

    customer_raw_df_cleaned=df.drop_duplicates(subset=['customer_unique_id'],keep='first').reset_index(drop=True)
    return customer_raw_df_cleaned

customer_raw_df=customer_cleaning(customer_raw_df)
print("After cleaning Total Unique customer : ",customer_raw_df['customer_unique_id'].nunique())
print("Total  cleaning customer Entry : ",len(customer_raw_df))

Total Unique customer :  96096
Total  customer Entry :  99441
After cleaning Total Unique customer :  96096
Total  cleaning customer Entry :  96096


In [8]:
# Product Raw Cleaning
product_raw_df=dfs['products_raw']
product_raw_df.head()
print(product_raw_df.isnull().sum())

def product_cleaning(df:pd.DataFrame)->pd.DataFrame:
    if df['product_id'].isnull().sum() > 0:
        df.dropna(subset=['product_id'])
    impute_cols = [
    'product_name_lenght',
    'product_description_lenght',
    'product_photos_qty'
]
    df[impute_cols] = df[impute_cols].fillna(0)

    physical_cols = [
    'product_weight_g',
    'product_length_cm',
    'product_height_cm',
    'product_width_cm'
]
    df['product_category_name'] = df['product_category_name'].fillna('not_specified')
    category_medians = df.groupby('product_category_name')[physical_cols].transform('median')

    for col in physical_cols:
        negative_mask = df[col] < 0
        negative_count = negative_mask.sum()

        if negative_count > 0:
            df.loc[negative_mask, col] = category_medians.loc[negative_mask, col]
        df[col] = df[col].fillna(category_medians[col])

    df_products=df.drop_duplicates(subset=['product_id'], keep='first')
    return df_products

product_raw_df=product_cleaning(product_raw_df)
print("After Cleaning : \n",product_raw_df.isnull().sum())

product_id                       0
product_category_name         1220
product_name_lenght           1220
product_description_lenght    1220
product_photos_qty            1220
product_weight_g                 4
product_length_cm                4
product_height_cm                4
product_width_cm                 4
dtype: int64
After Cleaning : 
 product_id                    0
product_category_name         0
product_name_lenght           0
product_description_lenght    0
product_photos_qty            0
product_weight_g              0
product_length_cm             0
product_height_cm             0
product_width_cm              0
dtype: int64


In [9]:
# category_trans_raw
category_trans_raw_df=dfs['category_trans_raw']
print(category_trans_raw_df.isnull().sum())
print(category_trans_raw_df.count())
print(category_trans_raw_df.nunique())

def category_cleaning(df:pd.DataFrame)->pd.DataFrame:
    df['product_category_name'] = df['product_category_name'].str.strip()
    df['product_category_name_english'] = df['product_category_name_english'].str.strip()

    initial_count = len(df)
    df_cat_trans_clean = df.drop_duplicates(
                                            subset=['product_category_name'],
                                            keep='first'
                        ).reset_index(drop=True)
    deleted_count = initial_count - len(df_cat_trans_clean)

    print(f"Deleted duplicate translations: {deleted_count}")
    print(f"Final unique translation count: {len(df_cat_trans_clean)}")
    return df_cat_trans_clean

category_trans_raw_df=category_cleaning(category_trans_raw_df)

product_category_name            0
product_category_name_english    0
dtype: int64
product_category_name            142
product_category_name_english    142
dtype: int64
product_category_name            71
product_category_name_english    71
dtype: int64
Deleted duplicate translations: 71
Final unique translation count: 71


In [10]:
# sellers raw
sellers_raw_df=dfs['sellers_raw']
sellers_raw_df.isnull().sum()
print(sellers_raw_df.count())
print(sellers_raw_df.nunique())

def cleaning_sellers(df:pd.DataFrame)->pd.DataFrame:
    df['seller_city'] = df['seller_city'].str.strip().str.title()
    df['seller_state'] = df['seller_state'].str.strip().str.upper()

    df_sellers_clean = df.drop_duplicates(
                            subset=['seller_id'],
                            keep='first'
                        ).reset_index(drop=True)
    initial_count = len(df)
    deleted_count = initial_count - len(df_sellers_clean)

    print(f"Deleted duplicate seller records: {deleted_count:,}")
    print(f"Final unique seller count: {len(df_sellers_clean):,}")
    return df_sellers_clean

sellers_raw_df=cleaning_sellers(sellers_raw_df)
print("After Cleaning")
print(sellers_raw_df.count())
print(sellers_raw_df.nunique())


seller_id                 6190
seller_zip_code_prefix    6190
seller_city               6190
seller_state              6190
dtype: int64
seller_id                 3095
seller_zip_code_prefix    2246
seller_city                611
seller_state                23
dtype: int64
Deleted duplicate seller records: 3,095
Final unique seller count: 3,095
After Cleaning
seller_id                 3095
seller_zip_code_prefix    3095
seller_city               3095
seller_state              3095
dtype: int64
seller_id                 3095
seller_zip_code_prefix    2246
seller_city                611
seller_state                23
dtype: int64


In [11]:
#Order Items Raw
order_items_raw_df=dfs['order_items_raw']
order_items_raw_df.isnull().sum()

def order_items_raw_cleaning(df:pd.DataFrame)->pd.DataFrame:
    CRITICAL_KEYS = ['order_id', 'product_id', 'seller_id', 'order_item_id']
    initial_count = len(df)
    df=df.dropna(
        subset=CRITICAL_KEYS,
    )
    rows_dropped_nan = initial_count - len(df)
    if rows_dropped_nan > 0:
        print(f"ℹ️ Dropped {rows_dropped_nan} rows due to missing critical IDs.")

    negative_price_count = (df['price'] <= 0).sum()
    negative_freight_count = (df['freight_value'] < 0).sum()
    if negative_price_count > 0 or negative_freight_count > 0:
        print(f"⚠️ ALERT: Found {negative_price_count} non-positive prices and {negative_freight_count} negative freight values.")
        df = df[
            (df['price'] > 0) &
            (df['freight_value'] >= 0)
        ].reset_index(drop=True)

    df_order_items_cleaned=df.drop_duplicates(
    subset=CRITICAL_KEYS,
    keep='first',
)
    rows_deduplicated = initial_count - len(df_order_items_cleaned)
    if rows_deduplicated > 0:
        print(f"ℹ️ Removed {rows_deduplicated} rows based on composite key duplication.")

    print(f"✅ Order Items Cleaned. Final Row Count: {len(df_order_items_cleaned):,}")
    return df_order_items_cleaned

order_items_raw_df=order_items_raw_cleaning(order_items_raw_df)
order_items_raw_df.count()



ℹ️ Removed 112650 rows based on composite key duplication.
✅ Order Items Cleaned. Final Row Count: 112,650


order_id               112650
order_item_id          112650
product_id             112650
seller_id              112650
shipping_limit_date    112650
price                  112650
freight_value          112650
dtype: int64

In [12]:
#Review Raw
review_raw_df=dfs['reviews_raw']
review_raw_df.isnull().sum()
review_raw_df.count()
review_raw_df[['review_id','order_id']].nunique()

def review_cleaning(df:pd.DataFrame)->pd.DataFrame:
    df=df.drop_duplicates(subset=['review_id'], keep='first')
    df['review_comment_title']=df['review_comment_title'].fillna("")
    df['review_comment_message']=df['review_comment_message'].fillna("")

    df_reviews_clean = df[[
    'review_id',
    'order_id',
    'review_score',
    'review_comment_message',
    'review_comment_title',
    'review_answer_timestamp'
]].copy()

    df_reviews_clean['review_score'] = df_reviews_clean['review_score'].astype('Int64')
    df_reviews_clean=df_reviews_clean.dropna(subset=['order_id'])
    df_reviews_clean=df_reviews_clean.dropna(subset=['review_score'])
    return df_reviews_clean

review_raw_df=review_cleaning(review_raw_df)
review_raw_df.count()

review_raw_df.nunique()

C:\Users\Ayush\AppData\Local\Temp\ipykernel_1936\217935004.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['review_comment_title']=df['review_comment_title'].fillna("")
C:\Users\Ayush\AppData\Local\Temp\ipykernel_1936\217935004.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['review_comment_message']=df['review_comment_message'].fillna("")


review_id                  98410
order_id                   98167
review_score                   5
review_comment_message     36153
review_comment_title        4525
review_answer_timestamp    98248
dtype: int64

In [13]:
# Geo Location Raw
geo_location_raw_df=dfs['geolocation_raw']
geo_location_raw_df.head()

def geo_location_cleaning(df:pd.DataFrame)->pd.DataFrame:
    df['geolocation_city'] = df['geolocation_city'].str.strip().str.title()
    df['geolocation_state'] = df['geolocation_state'].str.strip().str.upper()

    df = df.groupby('geolocation_zip_code_prefix').agg(
        customer_lat=('geolocation_lat', 'mean'),
        customer_lng=('geolocation_lng', 'mean'),
        customer_city=('geolocation_city', 'first'),
        customer_state=('geolocation_state', 'first')
    ).reset_index().rename(columns={'geolocation_zip_code_prefix': 'customer_zip_code_prefix'})
    missing_lat_lng = df[['customer_lat', 'customer_lng']].isnull().sum().sum()

    if missing_lat_lng > 0:
        df[['customer_lat', 'customer_lng']] = \
        df[['customer_lat', 'customer_lng']].fillna(df[['customer_lat', 'customer_lng']].median())
    return df

geo_location_raw_df=geo_location_cleaning(geo_location_raw_df)
geo_location_raw_df.count()


customer_zip_code_prefix    19015
customer_lat                19015
customer_lng                19015
customer_city               19015
customer_state              19015
dtype: int64

In [14]:
# Payment Raw
payments_raw_df=dfs['payments_raw']
payments_raw_df.isnull().sum()

def payments_cleaning(df:pd.DataFrame)->pd.DataFrame:
    negative_price_count = (df['payment_value'] <= 0).sum()
    if negative_price_count>0:
        df_payments = df[df['payment_value'] >= 0]
        df_payments.drop_duplicates(inplace=True)

    df_payments_agg = df_payments.groupby('order_id').agg(
    total_payment_value=('payment_value', 'sum'),
    max_payment_installments=('payment_installments', 'max'),
    payment_types_count=('payment_type', 'nunique')
).reset_index()
    return df_payments_agg

payments_raw_df=payments_cleaning(payments_raw_df)


In [15]:
def perform_final_merge()->pd.DataFrame:
    df_main=orders_raw_df.copy()
    df_main=df_main.merge(order_items_raw_df,on='order_id',how='left')

    df_main=df_main.merge(customer_raw_df,on='customer_id',how='left')
    df_main = df_main.merge(product_raw_df, on='product_id', how='left')

    df_main = df_main.merge(sellers_raw_df, on='seller_id', how='left')
    df_main = df_main.merge(
        category_trans_raw_df,
        on='product_category_name',
        how='left'
    ).drop(columns=['product_category_name'])

    df_main.rename(columns={'product_category_name_english': 'category_english'}, inplace=True)
    df_main = df_main.merge(review_raw_df, on='order_id', how='left')

    df_main = df_main.merge(payments_raw_df, on='order_id', how='left')
    print(df_main.columns)
    df_main = df_main.merge(
        geo_location_raw_df,
        left_on='customer_zip_code_prefix',
        right_on='customer_zip_code_prefix',
        how='left'
    )
    return df_main
df_main=perform_final_merge()


Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date',
       'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date',
       'price', 'freight_value', 'customer_unique_id',
       'customer_zip_code_prefix', 'customer_city', 'customer_state',
       'product_name_lenght', 'product_description_lenght',
       'product_photos_qty', 'product_weight_g', 'product_length_cm',
       'product_height_cm', 'product_width_cm', 'seller_zip_code_prefix',
       'seller_city', 'seller_state', 'category_english', 'review_id',
       'review_score', 'review_comment_message', 'review_comment_title',
       'review_answer_timestamp', 'total_payment_value',
       'max_payment_installments', 'payment_types_count'],
      dtype='object')


In [16]:
def finalize_silver_columns(df:pd.DataFrame)->pd.DataFrame:
    columns_to_drop = [
   'customer_id',
        'seller_zip_code_prefix',
        'order_status',
        'customer_city_x',
        'customer_state_x',
        'shipping_limit_date'
]
    df_master=df.drop(columns=columns_to_drop,errors='ignore')

    df_master=df_master.rename(columns={
    'customer_city_y': 'customer_city',
    'customer_state_y': 'customer_state',

    'price': 'item_price',
    'freight_value': 'shipping_cost',

    'review_score': 'review_score_stars',
    'review_comment_message': 'review_comment_text',

    'product_name_lenght': 'product_name_length',
    'product_description_lenght': 'product_description_length',

    'product_photos_qty': 'product_photo_qty',
    'product_weight_g': 'product_weight_grams'
})
    print("Renamed columns for final consumption layer.")
    return df_master
df_main=finalize_silver_columns(df_main)
df_main.columns
df_main.isnull().sum()


Renamed columns for final consumption layer.


order_id                            0
order_purchase_timestamp            0
order_approved_at                 161
order_delivered_carrier_date     1969
order_delivered_customer_date    3225
order_estimated_delivery_date       0
order_item_id                     775
product_id                        775
seller_id                         775
item_price                        775
shipping_cost                     775
customer_unique_id               4146
customer_zip_code_prefix         4146
product_name_length               775
product_description_length        775
product_photo_qty                 775
product_weight_grams              775
product_length_cm                 775
product_height_cm                 775
product_width_cm                  775
seller_city                       775
seller_state                      775
category_english                 2398
review_id                        1528
review_score_stars               1528
review_comment_text              1528
review_comme

In [17]:
def finalized_merged_clean(df_master:pd.DataFrame)->pd.DataFrame:
    CRITICAL_DROP_COLS = [
    'order_item_id',
    'product_id',
    'seller_id',
    'shipping_cost',
    'item_price'
]

    df_master=df_master.dropna(subset=CRITICAL_DROP_COLS)
    df_master['review_score_stars']=df_master['review_score_stars'].fillna(df_master['review_score_stars'].median())

    df_master['review_comment_text']=df_master['review_comment_text'].fillna("")
    df_master['review_comment_title']=df_master['review_comment_title'].fillna("")
    df_master=df_master.dropna(subset=['total_payment_value'])

    median_lat = df_master['customer_lat'].median()
    median_lng = df_master['customer_lng'].median()

    CRITICAL_CUSTOMER_COLS = ['customer_unique_id', 'customer_zip_code_prefix']
    df_master = df_master.dropna(subset=CRITICAL_CUSTOMER_COLS)
    location_cols = ['customer_lat', 'customer_lng', 'customer_city', 'customer_state']
    df_master['review_id'] = df_master['review_id'].fillna('UNKNOWN_MISSING')

    df_master['category_english'] = df_master['category_english'].fillna('not_specified')
    df_master['review_answer_timestamp'] = df_master['review_answer_timestamp'].fillna(
    df_master['order_delivered_customer_date']
)
    for col in location_cols:
        if 'lat' in col or 'lng' in col:
            df_master[col]=df_master[col].fillna(median_lat if 'lat' in col else median_lng)
        else:
            df_master[col]=df_master[col].fillna('Unknown')
    print("✅ Final Silver Imputation complete. Data is ready for Gold Aggregation.")
    return df_master
df_main=finalized_merged_clean(df_main)


C:\Users\Ayush\AppData\Local\Temp\ipykernel_1936\859695223.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_master['review_score_stars']=df_master['review_score_stars'].fillna(df_master['review_score_stars'].median())
C:\Users\Ayush\AppData\Local\Temp\ipykernel_1936\859695223.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_master['review_comment_text']=df_master['review_comment_text'].fillna("")
C:\Users\Ayush\AppData\Local\Temp\ipykernel_1936\859695223.py:14: SettingWithCopyWarning: 
A value

✅ Final Silver Imputation complete. Data is ready for Gold Aggregation.


In [37]:
# Saves as CSV FIle
df_main.to_csv('silver_main.csv', index=False)
# sql saving
load_temp_master_fact(df_main)